## 🚴‍♂️ Real-World Problem: Predicting Bicycle Rental Demand
 
 **Scenario: You work for a city bike rental company. 
            You want to predict how many bikes will be rented on a given day.**

Data features we can consider:

|Feature	  |     Meaning                           |          
|-------------|---------------------------------------|
|temperature  |	  Average daily temperature (°C)
|humidity	  |   Humidity (%)
|windspeed	  |    Wind speed (km/h)
|day_of_week  |	  Monday=0 … Sunday=6
|holiday	  |    1 if it’s a public holiday, 0 otherwise
|working_day  |	  1 if it’s a working day, 0 otherwise
|season	      |    1=Spring, 2=Summer, 3=Fall, 4=Winter
|random_noise |	  A completely random variable, just for testing

**Target: rentals → number of bikes rented that day.**

### step 1: create data

In [33]:
import pandas as pd
import numpy as np

n = 2000
np.random.seed(42)

# -----------------------------
# Numerical features
# -----------------------------
temperature = np.random.randint(14, 30, n)
humidity = np.random.randint(0, 56, n)
windspeed = np.random.randint(0, 18, n)

holiday = np.random.randint(0, 2, n)
working_day = np.random.randint(0, 2, n)

# -----------------------------
# Categorical feature
# -----------------------------
season = np.random.randint(1, 5, n)
day_of_week = np.random.randint(0, 7, n)

# -----------------------------
# Noise feature
# -----------------------------
random_noise = np.random.randn(n)

# -----------------------------
# Create dataframe
# -----------------------------
df = pd.DataFrame({
    'temperature': temperature,
    'humidity': humidity,
    'windspeed': windspeed,
    'day_of_week': day_of_week,
    'holiday': holiday,
    'working_day': working_day,
    'season': season,
    'random_noise': random_noise
})

# =====================================================
# ONE-HOT ENCODING: SEASON
# =====================================================
df["season_spring"] = (df["season"] == 1).astype(int)
df["season_summer"] = (df["season"] == 2).astype(int)
df["season_fall"]   = (df["season"] == 3).astype(int)
df["season_winter"] = (df["season"] == 4).astype(int)

# =====================================================
# ONE-HOT ENCODING: DAY OF WEEK
# =====================================================
df["Monday"]    = (df["day_of_week"] == 0).astype(int)
df["Tuesday"]   = (df["day_of_week"] == 1).astype(int)
df["Wednesday"] = (df["day_of_week"] == 2).astype(int)
df["Thursday"]  = (df["day_of_week"] == 3).astype(int)
df["Friday"]    = (df["day_of_week"] == 4).astype(int)
df["Saturday"]  = (df["day_of_week"] == 5).astype(int)
df["Sunday"]    = (df["day_of_week"] == 6).astype(int)

# -----------------------------
# Drop original categorical columns
# -----------------------------
df.drop(columns=["season", "day_of_week"], inplace=True)

# -----------------------------
# Target variable (rentals)
# -----------------------------
rentals = (

    200 * df["temperature"]
    -150 * df["humidity"]
    -100 * df["windspeed"]

    +500 * df["working_day"]
    +300 * df["holiday"]

    # season effects
    +600 * df["season_summer"]
    +300 * df["season_fall"]
    -500 * df["season_winter"]
    +200 * df["season_spring"]

    # weekday effects
    +50  * df["Tuesday"]
    +80  * df["Wednesday"]
    +60  * df["Thursday"]
    +70  * df["Friday"]
    +100 * df["Saturday"]
    +150 * df["Sunday"]

    # noise
    + np.random.randn(n) * 100
)

# -----------------------------
# Add target
# -----------------------------
df["rentals"] = rentals

print(df.head())

   temperature  humidity  windspeed  holiday  working_day  ...  Thursday  Friday  Saturday  Sunday      rentals
0           20        46         17        1            1  ...         0       0         0       1 -3269.034247
1           17        11          5        0            0  ...         0       0         1       0  1573.071175
2           26        15         11        0            0  ...         0       0         0       0  2140.178833
3           28        23         10        1            1  ...         0       0         0       0  2180.634753
4           24        18          1        0            0  ...         0       0         0       0  2289.262759

[5 rows x 18 columns]


### Step 2: Splitting data into train and test sets


In [24]:
from sklearn.model_selection import train_test_split

# features and target
X = df.drop(columns= "rentals")
y = df["rentals"]

# split dataset
X_train, X_test, y_train, y_test = train_test_split( X , y , test_size = 0.2, random_state = 42)

### Step 3: create evaluation function

In [25]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def evaluate(features):
    model = LinearRegression()
    
    # train only on selected features
    model.fit(X_train[features], y_train)
    
    # predict on unseen test data
    preds = model.predict(X_test[features])
    
    # return error on test set
    return mean_squared_error(y_test, preds)


### Step 4: forward selection



In [32]:
remaining = [

    # numerical features
    'temperature',
    'humidity',
    'windspeed',

    # binary features
    'holiday',
    'working_day',

    # season (one-hot encoded)
    'season_spring',
    'season_summer',
    'season_fall',
    'season_winter',

    # day of week (one-hot encoded)
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday',

    # noise feature
    'random_noise'
]

selected = []
prev_score = float('inf')

while remaining:
    scores = []

    #try each feature in remaining
    for f in remaining:
        
        current_features = selected + [f]
        score = evaluate(current_features)
        scores.append((score, f))

    #pick best feature
    scores.sort()
    best_score, best_feature = scores[0]

    #stop if no significant improvement in score
    if best_score >= prev_score:
        break
     
    #update staus
    selected.append(best_feature)
    remaining.remove(best_feature)
    prev_score = best_score
    print(f" Added: {best_feature}, MSE: {best_score:.2f}")


print("\n final selected features :", selected)

 Added: humidity, MSE: 1850147.12
 Added: temperature, MSE: 488950.23
 Added: windspeed, MSE: 219636.95
 Added: season_winter, MSE: 133452.27
 Added: working_day, MSE: 58237.66
 Added: holiday, MSE: 29281.71
 Added: season_summer, MSE: 16247.23
 Added: Sunday, MSE: 12685.20
 Added: season_spring, MSE: 9776.28
 Added: Saturday, MSE: 9342.46
 Added: random_noise, MSE: 9083.41
 Added: Wednesday, MSE: 8884.46
 Added: Thursday, MSE: 8627.04
 Added: season_fall, MSE: 8627.04

 final selected features : ['humidity', 'temperature', 'windspeed', 'season_winter', 'working_day', 'holiday', 'season_summer', 'Sunday', 'season_spring', 'Saturday', 'random_noise', 'Wednesday', 'Thursday', 'season_fall']


---

### IMPORTANT CONCEPTS



#### 🚴 1. Problem Statement

We created a synthetic Machine Learning problem:

```text
Predict bike rentals per day
```

using features such as:
- temperature
- humidity
- windspeed
- season
- holidays
- day of week

This is a:
- supervised learning problem
- regression problem

because target variable (`rentals`) is numeric.

---

#### 🧠 2. Goal of This Project

Purpose of this project:

- understand regression intuitively
- learn feature engineering
- understand categorical variables
- practice one-hot encoding
- implement forward selection manually
- observe behavior of noise features
- build ML intuition

---

#### 🔢 3. Numerical Features

Numerical features are already numeric.

Examples:

```python
temperature
humidity
windspeed
```

These can directly be used in Linear Regression.

---

#### 🔁 4. Binary Features

Binary features already contain:
- 0
- 1

Examples:

```python
holiday
working_day
```

No extra processing required.

---

#### 🏷️ 5. Categorical Features

Categorical features contain labels/categories.

Examples:

```python
season
day_of_week
```

Example values:
- summer
- winter
- Monday
- Friday

Problem:
Regression models cannot directly understand categories.

---

#### 🔄 6. One-Hot Encoding

To use categorical variables in regression,
we convert them into binary columns using:

```text
One-Hot Encoding
```

---

#### Example

Original feature:

| season |
|--------|
| summer |

After encoding:

| season_spring | season_summer | season_fall | season_winter |
|---|---|---|---|
| 0 | 1 | 0 | 0 |

Meaning:
- 1 → category active
- 0 → category inactive

---

#### 🧱 7. Dummy Variables

Each one-hot encoded column is called a:

```text
Dummy Variable
```

Examples:
- season_summer
- Monday
- Sunday

These are simply YES/NO indicators.

---

#### ⚠️ 8. Multicollinearity

Problem occurs when ALL dummy variables are included.

Example:

```text
Mon + Tue + Wed + Thu + Fri + Sat + Sun = 1
```

Meaning:
- one variable can always be predicted from others
- features become dependent

This creates:

```text
Multicollinearity
```

---

#### Why Multicollinearity is Bad

It causes:
- redundant information
- unstable coefficients
- infinite valid solutions
- difficulty interpreting weights

Regression expects features to provide independent information.

---

#### 🎯 9. Baseline Category

To avoid multicollinearity:
we drop ONE dummy variable from each category group.

Examples:
- drop Monday
- drop Spring

These become:

```text
Baseline Categories
```

---

#### Interpretation

Now model learns:
- Tuesday effect relative to Monday
- Summer effect relative to Spring

If all dummy variables = 0,
baseline category is active.

---

#### 🧪 10. Synthetic Dataset

We manually generated fake data using NumPy.

Purpose:
- safely experiment with ML concepts
- control feature relationships
- observe model behavior

---

#### 🎲 11. Random Noise Feature

We added:

```python
random_noise
```

Purpose:
- simulate useless feature
- test feature selection methods

This feature has NO true relationship with target.

---

#### 🌪️ 12. Noise in Target Variable

We also added randomness to target variable:

```python
+ np.random.randn(n) * 500
```

Purpose:
- simulate real-world unpredictability
- prevent perfect prediction
- make dataset realistic

---

#### 📊 13. Target Variable

We manually created target variable:

```text
rentals = weighted sum of features + noise
```

Example:
- temperature → positive effect
- humidity → negative effect
- summer → positive effect
- winter → negative effect

This gives known relationships for experimentation.

---

#### ✂️ 14. Train-Test Split

Initially:
- training and testing were done on same data

Problem:
- model memorized data
- training error always decreased
- all features got selected

Solution:
Use train-test split.

---

## Purpose of Train-Test Split

### Train Set
Used for learning patterns.

### Test Set
Used for evaluating generalization.

This prevents:
- memorization
- misleading evaluation

---

#### 📉 15. Evaluation Metric — MSE

We used:

```text
Mean Squared Error (MSE)
```

Formula idea:

```text
(actual - predicted)^2
```

Then averaged over all samples.

---

#### Interpretation

Lower MSE:
- predictions closer to true values
- better model performance

---

#### 🔍 16. Forward Selection

Forward Selection is a feature selection method.

Process:

```text
Start with no features
↓
Try each remaining feature
↓
Select feature with lowest MSE
↓
Repeat
```

Goal:
find useful subset of features.

---

#### 🤖 17. Greedy Algorithm

Forward Selection is a:

```text
Greedy Algorithm
```

Meaning:
At each step:
- choose best immediate option
- ignore future/global consequences

---

#### Important Property

Forward Selection only asks:

```text
Did this feature reduce error?
```

It does NOT ask:
- Is this feature truly meaningful?
- Is this feature causal?
- Will this generalize perfectly?

---

#### ⚠️ 18. Why random_noise Got Selected

Even though random_noise is useless,
it still got selected sometimes.

Why?

Because:
- finite datasets create accidental correlations
- random features may slightly improve MSE
- greedy algorithms accept local improvements

---

#### 🧠 19. Important Statistical Insight

Random noise is useless:

```text
IN EXPECTATION
```

But:
in a finite sample,
it can accidentally correlate with target.

Therefore:

```text
Selected feature ≠ truly important feature
```

This is normal behavior in Machine Learning.

---

#### 📈 20. Dataset Size Effect

Larger datasets:
- reduce accidental correlations
- improve stability
- reduce overfitting

Smaller datasets:
- noisier
- less stable
- easier to overfit

---

#### ⚖️ 21. Signal vs Noise

Strong signal features:
- temperature
- humidity

Weak/noisy features:
- random_noise

Feature selection tries to separate:
- meaningful patterns
- accidental randomness

This process is imperfect.

---

#### 🚀 22. Key Learning Outcomes

This project covered:

- regression
- synthetic datasets
- categorical variables
- one-hot encoding
- dummy variables
- multicollinearity
- baseline categories
- train-test split
- MSE
- feature selection
- forward selection
- greedy algorithms
- noise features
- accidental correlations
- overfitting intuition

---

#### 🧠 Final Intuition

Machine Learning models do NOT discover absolute truth.

They learn patterns from:
- finite data
- noisy observations
- imperfect samples

Therefore:

```text
Good prediction ≠ true causality
```

and

```text
Selected feature ≠ guaranteed important feature
```

Feature selection is probabilistic, not magical.